# Original benchmark and canonical-family robustness

Formal DiffJev workflow; see AGENTS.md and prompt.md.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import sys
from pathlib import Path

# 与 notebook00 使用同一个项目目录
ROOT = Path("/content/drive/MyDrive/DiffJev")
assert (ROOT / "AGENTS.md").is_file(), f"项目目录不存在：{ROOT}"

os.chdir(ROOT)
os.environ["DIFFJEV_ROOT"] = str(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from diffjev.notebook_runtime import NotebookRuntime, install_environment

ENV_DIR = Path("/content/diffjev-env")
WORKER = ENV_DIR / "bin/python"

if not WORKER.is_file():
    # 新运行时：只重建依赖环境，不下载模型、不重新准备数据
    runtime = install_environment(ROOT, ENV_DIR)
else:
    runtime = NotebookRuntime(ROOT, WORKER)
    runtime.save_settings()

print("项目目录：", ROOT)
print("实验 Python：", runtime.python)

In [ ]:
import os, sys, gc
from pathlib import Path
ROOT = Path(os.environ.get('DIFFJEV_ROOT', Path.cwd())).resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'AGENTS.md').exists() and Path('/content/drive/MyDrive/DiffJev/AGENTS.md').exists():
    ROOT = Path('/content/drive/MyDrive/DiffJev')
assert (ROOT / 'AGENTS.md').exists(), 'Mount Drive and set DIFFJEV_ROOT to the checkout used by notebook 00'
sys.path.insert(0, str(ROOT / 'src'))
from diffjev.notebook_runtime import NotebookRuntime
runtime = NotebookRuntime(ROOT)
runtime.python_code("from diffjev.preparation import check_environment; from pathlib import Path; check_environment(Path.cwd())", label='04_exp1_exp2_environment')
def run_python(source):
    prefix = '''import gc
from pathlib import Path
from diffjev import workflows as w
from diffjev.config import load_yaml_config
ROOT = w.ROOT
cfg = load_yaml_config(str(ROOT / 'configs/workflow.yaml'))
SEED, DEVICE = cfg['seed'], cfg['device']
'''
    runtime.python_code(prefix + source, label='04_exp1_exp2')
print('Experiment worker:', runtime.python)


In [ ]:
run_python(r"""
for backbone in cfg['backbones']:
    runner = w.load_runner(backbone, seed=SEED, device=DEVICE)
    output = w.run_directory(backbone, seed=SEED)/'evaluation'
    w.evaluate_benchmark(runner, output, robustness=backbone=='llada_8b_base')
    del runner
    gc.collect()
""")

In [ ]:
run_python(r"""
runner = w.load_runner('llada_8b_base', method='serialized_dlm', seed=SEED, device=DEVICE)
w.evaluate_benchmark(runner, w.run_directory('llada_8b_base','serialized_dlm',seed=SEED)/'evaluation')
del runner
gc.collect()
""")

In [ ]:
run_python(r"""
names = cfg['references'] + (['jev_api'] if cfg['enable_paid_api'] else [])
for name in names:
    runner = w.baseline_runner(name, device=DEVICE, allow_api=cfg['enable_paid_api'])
    w.evaluate_benchmark(runner, ROOT/'outputs/formal/references'/name)
    del runner
    gc.collect()
""")

The original revision-pinned official evaluator writes official/scores.json. Project summaries are labeled separately because the original scorer defines task-specific headlines, not a single universal aggregate. Probability metrics use the same candidate-only normalization as forced-choice predictions.